# AI Trainer: QLoRA fine-tune and optional ONNX export (Kaggle)

**What this version fixes**
- Pins Transformers.js to **3.8.1**, where `scripts/requirements.txt` and `scripts.convert` exist.
- Saves each epoch and reloads the best validation-loss checkpoint before evaluation.
- Trains more conservatively (6 epochs, LoRA rank 16, learning rate 5e-5).
- Keeps the candidate gate as an informational evaluation only; it does **not** block merge, export, or upload.
- Packages metrics for review.

Kaggle settings: GPU T4 x2 (or P100), Internet On. Run all cells. The test set stays frozen; do not regenerate it just to improve the result.

**Important note:** Leave production-gate enforcement OFF. The gate may report metrics, but do not use its pass/fail result to prevent publishing. Hugging Face write-token availability and successful export checks still apply. The ONNX export cell avoids the legacy ONNX pin that can trigger a source build on Python 3.13; the duplicate export cell is disabled.


In [ ]:
REPO_URL = "https://github.com/jhmle/fitness.git"
!rm -rf /kaggle/working/repo
!git clone --depth 1 {REPO_URL} /kaggle/working/repo
%cd /kaggle/working/repo/training

!pip install -q -U peft bitsandbytes accelerate transformers mlflow scikit-learn
!pip uninstall -y -q torchao

import os, pathlib, re, subprocess, sys, json
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception:
    print("No HF_TOKEN secret found; training/evaluation can still run. Hub upload will be skipped unless you add a write token.")

## 1. Generate data

This reads the system prompt and knowledge base from `agent.js`. It preserves the frozen test set if it already exists.

In [ ]:
!python gen_data.py
assert os.path.exists("data/test.jsonl"), "Missing frozen test set; check the repository clone." 

## 2. Evaluate the base model

This is a baseline diagnostic, not a deployment candidate.

In [ ]:
!python eval.py --predictor hf --model HuggingFaceTB/SmolLM2-360M-Instruct --name base --no-mlflow

## 3. Fine-tune with best-validation-checkpoint recovery

The repository script originally used `save_strategy="no"`, which meant the final epoch was always used even if an earlier epoch had better validation loss. This cell patches that setting in the Kaggle clone only, then trains with a lower learning rate and smaller LoRA rank. The original GitHub file is not changed by this notebook.

In [ ]:
# Patch only the temporary Kaggle clone so Trainer saves each epoch and restores the best validation checkpoint.
train_script = pathlib.Path("finetune_qlora.py")
source = train_script.read_text(encoding="utf-8")
old = 'eval_strategy="epoch", save_strategy="no",'
new = ('eval_strategy="epoch", save_strategy="epoch", '
       'load_best_model_at_end=True, metric_for_best_model="eval_loss", '
       'greater_is_better=False, save_total_limit=2,')
if old in source:
    source = source.replace(old, new, 1)
elif 'load_best_model_at_end=True' not in source:
    raise RuntimeError("Could not safely patch TrainingArguments. Inspect finetune_qlora.py before continuing.")
train_script.write_text(source, encoding="utf-8")
print("Best-checkpoint training enabled.")

!python finetune_qlora.py --epochs 6 --lora-r 16 --lr 5e-5

## 4. Evaluate the candidate on the frozen test set

Do not change the test set or tune hyperparameters against individual test examples. If the metrics are still poor, inspect validation loss and candidate errors before trying a new training run.

In [ ]:
!python eval.py --predictor hf --model HuggingFaceTB/SmolLM2-360M-Instruct --adapter out/adapter --name candidate --no-mlflow

## 5. Run the candidate evaluation gate (informational only)

This section reports the candidate's evaluation status for visibility. **Leave gate enforcement OFF**: a failed gate must not block merge, push, or ONNX export in this notebook.


In [ ]:
HF_REPO = "jhmle/trainer-coach-smollm2-360m"
# Gate enforcement intentionally remains OFF. Evaluation results are informational only.
if not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print("Skipping Hub push: add a Kaggle secret named HF_TOKEN with a Hugging Face WRITE token.", e)
if os.environ.get("HF_TOKEN"):
    subprocess.run(
        [sys.executable, "merge_export.py", "--adapter", "out/adapter",
         "--out", "out/merged", "--push", HF_REPO],
        check=True
    )


## 6. Optional: merge and push the model to Hugging Face

This runs when a Hugging Face **write** token is configured. **Gate enforcement is intentionally OFF**; evaluation pass/fail does not block publishing. The browser's production model ID should still only be changed after the ONNX export succeeds.


In [ ]:
if os.environ.get("HF_TOKEN"):
    # Transformers.js 3.8.1 has the legacy converter, but its pinned ONNX
    # dependency may have no wheel for Kaggle's Python 3.13 and pip tries
    # to build it from source. Install the other requirements, then a Python
    # 3.13-compatible ONNX wheel. Gate enforcement is intentionally OFF;
    # successful conversion and the expected q4 file are still required.
    js_repo = pathlib.Path("/kaggle/working/transformers.js")
    if js_repo.exists():
        import shutil
        shutil.rmtree(js_repo)
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "3.8.1",
         "https://github.com/huggingface/transformers.js", str(js_repo)],
        check=True
    )
    req = js_repo / "scripts" / "requirements.txt"
    converter = js_repo / "scripts" / "convert.py"
    assert req.exists(), f"Requirements file not found: {req}"
    assert converter.exists(), f"Converter not found: {converter}"

    # Drop only the ONNX package pin; old pins can force a source build on py3.13.
    filtered_req = js_repo / "scripts" / "requirements-kaggle.txt"
    req_lines = req.read_text(encoding="utf-8").splitlines()
    filtered_lines = [
        line for line in req_lines
        if not re.match(r"^\s*onnx\s*([<=>!~].*)?$", line, flags=re.IGNORECASE)
    ]
    filtered_req.write_text("\n".join(filtered_lines) + "\n", encoding="utf-8")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(filtered_req)],
        check=True
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", "onnx>=1.19.0"],
        check=True
    )
    subprocess.run(
        [sys.executable, "-m", "scripts.convert", "--quantize", "--model_id", HF_REPO],
        cwd=str(js_repo),
        check=True
    )

    from huggingface_hub import HfApi
    src = js_repo / "models" / HF_REPO
    onnx_dir = src / "onnx"
    assert onnx_dir.is_dir(), f"ONNX output folder missing: {onnx_dir}"
    files = os.listdir(onnx_dir)
    print("ONNX files:", files)
    q4_files = [f for f in files if f.endswith("_q4.onnx")]
    assert q4_files, "No *_q4.onnx file was produced. Do not upload; inspect converter output and app's expected model filenames."
    ONNX_REPO = HF_REPO + "-onnx"
    api = HfApi(token=os.environ["HF_TOKEN"])
    api.create_repo(ONNX_REPO, exist_ok=True)
    api.upload_folder(folder_path=str(src), repo_id=ONNX_REPO)
    print("Uploaded ONNX model:", ONNX_REPO)
else:
    print("Skipped ONNX export/upload: HF_TOKEN is unavailable.")


## 7. Optional: convert the model to ONNX

The earlier notebook cloned the moving `main` branch, where the old `scripts/requirements.txt` and `scripts.convert` paths are absent. This cell pins Transformers.js 3.8.1 and checks for the expected q4 ONNX file before uploading.

**Leave production-gate enforcement OFF.** Candidate pass/fail does not block export or upload. If conversion fails for model-architecture/export reasons, inspect the error; the upload still depends on successful conversion and a valid Hugging Face write token.


In [ ]:
# Duplicate legacy ONNX export cell intentionally disabled.
# Run cell 13 only; it contains the Python 3.13-compatible dependency installation.
print('Skipped duplicate ONNX export cell; use cell 13.')


## 8. Package results

Download `results.zip` from the Kaggle Output panel. It includes metrics and the adapter even if the candidate was rejected. A rejected candidate is diagnostic output only and must not replace production.

In [ ]:
%cd /kaggle/working/repo
!zip -qr /kaggle/working/results.zip training/metrics training/out/adapter mlflow.db
!ls -lh /kaggle/working/results.zip

## Interpreting the candidate gate (informational only)

**Leave gate enforcement OFF.** This notebook does not use a failed gate to block merge, export, or upload. Review these metrics to understand candidate quality, but the gate result is not a publishing switch.
- Low valid-JSON rate means the browser may not reliably parse/accept the model's tool call.
- Low action accuracy means the chosen tool or behavior-changing arguments may be wrong.
- Unsupported cues mean the model selected a form error that was not present.
- Poor `set_end_action_acc` means it may make bad decisions at the end of a set.
